In [69]:
import pandas as pd

ca = pd.read_csv("/Users/jli624/Desktop/cmpt733/project/ca_gas_price_precleaned.csv")
us = pd.read_csv("/Users/jli624/Desktop/cmpt733/project/us_gas_price_precleaned.csv")
mx = pd.read_csv("/Users/jli624/Desktop/cmpt733/project/mx_gas_price_precleaned.csv")
xrate = pd.read_csv("/Users/jli624/Desktop/cmpt733/project/xrate.csv")

us = us.rename(columns={"Date": "date"})
mx = mx.rename(columns={"Date": "date"})
ca = ca.rename(columns={"Date": "date"})

In [70]:
us["date"] = pd.to_datetime(us["date"], format="%b-%Y").dt.strftime("%Y-%m")

us = us.rename(columns={
    "U.S. No 2 Diesel Retail Prices (Dollars per Gallon)": "usd"
})

In [71]:
ca = ca[ca["Type of fuel"].str.contains("Diesel", case=False)]

ca = (
    ca.groupby("REF_DATE")["VALUE"]
    .mean()
    .reset_index()
)

ca = ca.rename(columns={"REF_DATE": "date"})
ca = ca.rename(columns={"VALUE": "cad"})
ca["cad"] = ca["cad"] / 100

In [72]:
mx["date"] = mx["date"].str[:7]

mx = mx.rename(columns={"Diesel_MXN_L": "mxn"})

In [73]:
xrate["date"] = xrate["date"].str[:7]
xrate.head()

,date,MXN_CAD,USD_CAD
0,2016-01,0.07871,1.4226
1,2016-02,0.07486,1.3788
2,2016-03,0.07499,1.3210
3,2016-04,0.07332,1.2818
4,2016-05,0.07136,1.2943


In [74]:
merged_all = (
    xrate
    .merge(us, on="date", how="inner")
    .merge(mx, on="date", how="inner")
    .merge(ca, on="date", how="inner")
)

# US: USD/gallon → CAD/L
merged_all["us_cad_l"] = (
    merged_all["usd"] * merged_all["USD_CAD"] / 3.78541
)

# Mexico: MXN/L → CAD/L
merged_all["mx_cad_l"] = (
    merged_all["mxn"] * merged_all["MXN_CAD"]
)

# Canada already CAD/L
merged_all["ca_cad_l"] = merged_all["cad"]

final_prices = merged_all[["date", "us_cad_l", "mx_cad_l", "ca_cad_l"]]

final_prices["integrated_gas_price"] = (
    0.7 * final_prices["us_cad_l"] +
    0.2 * final_prices["mx_cad_l"] +
    0.1 * final_prices["ca_cad_l"]
)

final_prices.head()


/var/folders/7j/4d_gnbqx14s1px8z4_03t8p40000gn/T/ipykernel_45963/4019113103.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_prices["integrated_gas_price"] = (


,date,us_cad_l,mx_cad_l,ca_cad_l,integrated_gas_price
0,2016-01,0.805364,1.083837,0.947000,0.875222
1,2016-02,0.727753,1.051034,0.905706,0.810204
2,2016-03,0.729350,1.073857,0.926647,0.817981
3,2016-04,0.728701,1.069739,0.918647,0.815903
4,2016-05,0.791540,1.060410,0.953294,0.861490


In [75]:
final_prices.to_csv(
    "/Users/jli624/Desktop/cmpt733/project/gas_price.csv",
    index=False
)